# Aprendizado de Máquina — Aula prática 08

## Métricas para Classificação

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

A Aula 07 terminou com uma tabela de acurácias e um incômodo: nenhuma daquelas
linhas dizia **que tipo** de erro cada modelo cometia. Este notebook é sobre isso, e
sobre uma frase que vai se repetir:

> **um classificador entrega uma probabilidade; a classe é uma decisão que você
> toma depois, e a métrica tem de refletir o que essa decisão custa.**

Vamos trabalhar com o `bank_train_redux.csv`: 66 mil clientes, 200 variáveis
anônimas e uma classe positiva que aparece em **10%** dos casos. É nesse regime que
a acurácia deixa de significar qualquer coisa — e, não por acaso, é o regime de
quase todo problema interessante: fraude, diagnóstico, churn, inadimplência.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- mostrar por que a acurácia é inútil quando uma classe é rara;
- ler uma matriz de confusão e calcular precisão, revocação e $F_1$ à mão;
- **derivar e verificar** o corte ótimo quando os dois erros têm custos diferentes;
- interpretar a AUC como uma probabilidade — e confirmá-la por simulação;
- escolher entre curva ROC e curva precisão–revocação conforme a prevalência;
- distinguir **ordenar bem** de **estimar bem**, medindo calibração e Brier;
- passar o `scoring` certo para a validação cruzada.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os objetos novos vêm quase todos de `sklearn.metrics`. O `calibration_curve` mora
em `sklearn.calibration`, e é ele que responde à pergunta da Seção 9.

In [ ]:
import sklearn.model_selection as skm
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             classification_report, confusion_matrix,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. A base, e a acurácia que não diz nada

Primeiro a leitura, com a limpeza que a Aula 06 diagnosticou: o arquivo veio com
separadores sobrando na última coluna.

In [ ]:
import os

_nome = "bank_train_redux.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../../recursos/dados/
_lugares = [_nome, os.path.join("..", "..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"nao encontrei '{_nome}'. Procurei nesta pasta e em "
        "../../recursos/dados/. Ponha o .csv ao lado deste notebook, "
        "ou mude o caminho se for necessário."
    )

banco = pd.read_csv(_caminho, nrows=40_000)
banco = banco.replace(to_replace=";", value="", regex=True)
banco = banco.rename(columns={"var_199;;;;;;;": "var_199"})
banco["var_199"] = pd.to_numeric(banco["var_199"])

Xb = banco.drop(columns=["ID_code", "target"]).astype(float).values
yb = banco["target"].values
print(f"{Xb.shape[0]} clientes, {Xb.shape[1]} variaveis")
print(f"prevalencia da classe positiva: {yb.mean():.4f}")

X_tr, X_te, y_tr, y_te = skm.train_test_split(Xb, yb, test_size=0.3,
                                              random_state=0, stratify=yb)

In [ ]:
modelo = Pipeline([("sc", StandardScaler()),
                   ("lg", LogisticRegression(max_iter=2000))]).fit(X_tr, y_tr)
p = modelo.predict_proba(X_te)[:, 1]

acuracia = (modelo.predict(X_te) == y_te).mean()
trivial = 1 - y_te.mean()
print(f"acuracia do modelo                       : {acuracia:.4f}")
print(f"acuracia de 'ninguem e' positivo'        : {trivial:.4f}")
print(f"ganho sobre nao fazer nada               : {acuracia - trivial:+.4f}")
print(f"\npositivos que o modelo encontrou: "
      f"{int(((modelo.predict(X_te) == 1) & (y_te == 1)).sum())} de {int(y_te.sum())}")

Aqui está o problema inteiro, em três linhas. O modelo tem 90% de acurácia; um
classificador que responde *"ninguém é positivo"*, sem olhar variável nenhuma, tem
praticamente a mesma. E o modelo encontra uma fração pequena dos positivos.

A acurácia é a média de duas taxas de acerto **ponderada pela prevalência**. Com
uma classe a 10%, ela é essencialmente a taxa de acerto na classe majoritária, e o
que acontece com a minoria quase não entra na conta. Se a sua pergunta é sobre a
minoria — e em fraude, diagnóstico ou inadimplência é sempre —, a acurácia responde
a outra pergunta.

---
## 3. Matriz de confusão, e as métricas que saem dela

Toda métrica de classificação binária é uma função de quatro números.

In [ ]:
pred = (p >= 0.5).astype(int)
mc = confusion_matrix(y_te, pred)
vn, fp, fn, vp = mc.ravel()

print(pd.DataFrame(mc, index=["real 0", "real 1"],
                   columns=["previu 0", "previu 1"]).to_string())
print(f"\nverdadeiros negativos (VN): {vn:6d}")
print(f"falsos positivos      (FP): {fp:6d}")
print(f"falsos negativos      (FN): {fn:6d}")
print(f"verdadeiros positivos (VP): {vp:6d}")

In [ ]:
precisao = vp / (vp + fp) if vp + fp else np.nan
revocacao = vp / (vp + fn)
f1 = 2 * precisao * revocacao / (precisao + revocacao)
especificidade = vn / (vn + fp)

print(f"precisao   VP/(VP+FP) = {precisao:.4f}   dos que chamei de positivo, quantos eram")
print(f"revocacao  VP/(VP+FN) = {revocacao:.4f}   dos positivos que existiam, quantos achei")
print(f"F1         (media harmonica) = {f1:.4f}")
print(f"especificidade VN/(VN+FP) = {especificidade:.4f}")
print(f"acuracia   (VP+VN)/n  = {(vp+vn)/len(y_te):.4f}\n")
print(classification_report(y_te, pred, digits=3))

As duas primeiras respondem a perguntas diferentes, e confundi-las é o erro mais
comum da área:

- **precisão**: *dos que eu chamei de positivo, quantos eram mesmo?* — é a pergunta
  de quem vai **agir** sobre a previsão (ligar para o cliente, abrir a investigação);
- **revocação**: *dos positivos que existiam, quantos eu encontrei?* — é a pergunta
  de quem teme **deixar passar** (o tumor, a fraude).

As duas se opõem: chamar mais gente de positivo aumenta a revocação e derruba a
precisão. O $F_1$ é a média harmônica delas, e a média harmônica é dura com o
desequilíbrio — basta uma das duas ser pequena para o $F_1$ ser pequeno.

---
## 4. O corte é um botão, não uma constante

O $0{,}5$ não veio de lugar nenhum. Vamos girar o botão e ver as três métricas se
mexerem.

In [ ]:
cortes = np.linspace(0.02, 0.95, 200)
prec, rec, efe = [], [], []
for t in cortes:
    q = p >= t
    tp = np.sum(q & (y_te == 1)); fp_ = np.sum(q & (y_te == 0))
    fn_ = np.sum(~q & (y_te == 1))
    pr = tp / (tp + fp_) if tp + fp_ else np.nan
    rc = tp / (tp + fn_)
    prec.append(pr); rec.append(rc)
    efe.append(2 * pr * rc / (pr + rc) if pr and rc else np.nan)
prec, rec, efe = np.array(prec), np.array(rec), np.array(efe)

t_f1 = cortes[np.nanargmax(efe)]
print(f"F1 maximo: {np.nanmax(efe):.4f} no corte {t_f1:.3f}")
print(f"F1 no corte 0,5: {efe[np.argmin(np.abs(cortes - 0.5))]:.4f}")

fig, ax = subplots(figsize=(5.6, 3.2))
ax.plot(cortes, prec, label="precisao")
ax.plot(cortes, rec, label="revocacao")
ax.plot(cortes, efe, label="F1", lw=2)
ax.axvline(0.5, ls=":", color="gray", label="corte padrao (0,5)")
ax.axvline(t_f1, ls="--", color="green", label=f"F1 maximo ({t_f1:.2f})")
ax.set_xlabel("corte"); ax.set_ylabel("metrica"); ax.legend(fontsize=7.5)

O corte que maximiza o $F_1$ não é $0{,}5$ — e não tinha por que ser. O $0{,}5$
minimiza o **erro total**, que é a acurácia disfarçada, e já vimos o que ela vale
aqui.

---
## 5. Custos assimétricos: o corte ótimo tem fórmula

Suponha que um falso negativo custa $c_{FN}$ e um falso positivo custa $c_{FP}$.
O custo esperado de chamar de positivo uma observação com probabilidade $p$ é
$(1-p)\,c_{FP}$; o de chamá-la de negativo é $p\,c_{FN}$. Vale a pena chamar de
positivo quando

$$(1-p)\,c_{FP} < p\,c_{FN}
  \qquad\Longleftrightarrow\qquad
  p > \frac{c_{FP}}{c_{FP} + c_{FN}} .$$

**O corte ótimo depende só da razão entre os custos** — não dos dados, não da
prevalência, não do modelo. É uma das fórmulas mais úteis desta área, e dá para
conferir numericamente.

In [ ]:
def custo_total(t, c_fp, c_fn):
    q = p >= t
    return c_fp * np.sum(q & (y_te == 0)) + c_fn * np.sum(~q & (y_te == 1))


grade_t = np.linspace(0.005, 0.995, 400)
print(f"{'c_FP':>6} {'c_FN':>6} {'corte teorico':>15} {'corte medido':>14}")
for c_fp, c_fn in [(1, 1), (1, 5), (1, 20), (5, 1)]:
    teorico = c_fp / (c_fp + c_fn)
    custos = [custo_total(t, c_fp, c_fn) for t in grade_t]
    medido = grade_t[int(np.argmin(custos))]
    print(f"{c_fp:6d} {c_fn:6d} {teorico:15.3f} {medido:14.3f}")

In [ ]:
c_fp, c_fn = 1, 20
custos = np.array([custo_total(t, c_fp, c_fn) for t in grade_t])

fig, ax = subplots(figsize=(5.4, 3.0))
ax.plot(grade_t, custos / len(y_te))
ax.axvline(c_fp / (c_fp + c_fn), ls="--", color="green",
           label=f"corte teorico = {c_fp/(c_fp+c_fn):.3f}")
ax.axvline(0.5, ls=":", color="gray", label="corte padrao")
ax.set_xlabel("corte"); ax.set_ylabel("custo medio por observacao")
ax.set_title(f"um falso negativo custa {c_fn} falsos positivos", fontsize=9)
ax.legend(fontsize=8)

economia = 1 - custos[np.argmin(custos)] / custo_total(0.5, c_fp, c_fn)
print(f"usar o corte otimo em vez de 0,5 reduz o custo em {economia:.1%}")

> **A lição.** O corte medido acompanha o teórico em todos os cenários — a
> diferença que sobra é a granularidade da grade e o fato de o custo empírico ser
> uma escada. E o deslocamento é grande: quando um falso negativo custa 20 falsos positivos, o corte
> ótimo é $1/21 \approx 0{,}048$, não $0{,}5$.
>
> Isso reorganiza a divisão de trabalho. **Estimar $P(Y=1\mid x)$ é problema de
> estatística; escolher o corte é problema de negócio.** Se você não sabe os custos,
> não invente um corte — entregue a probabilidade e deixe quem sabe decidir. E se
> alguém lhe pedir "o classificador", pergunte quanto custa cada tipo de erro.

Agora com números de verdade: a ação sobre um positivo previsto custa R$ 30, e um
positivo não detectado custa R$ 900. A fórmula dá o corte; os dados dizem quantas
pessoas isso mobiliza.

In [ ]:
c_fp_r, c_fn_r = 30, 900
corte_teorico = c_fp_r / (c_fp_r + c_fn_r)

custos_r = np.array([custo_total(t, c_fp_r, c_fn_r) for t in grade_t])
corte_medido = grade_t[int(np.argmin(custos_r))]

marcados = p >= corte_medido
vp_r = int(np.sum(marcados & (y_te == 1)))
fp_r = int(np.sum(marcados & (y_te == 0)))
fn_r = int(np.sum(~marcados & (y_te == 1)))
n_te = len(y_te)

print(f"corte teorico {corte_teorico:.4f}   corte medido na grade {corte_medido:.4f}")
print(f"\npor mil clientes do conjunto de teste:")
print(f"   investigados : {1000 * marcados.sum() / n_te:.0f}")
print(f"   positivos encontrados : {1000 * vp_r / n_te:.1f}  "
      f"de {1000 * (vp_r + fn_r) / n_te:.1f} existentes")
print(f"   falsos positivos      : {1000 * fp_r / n_te:.0f}")
print(f"\nrevocacao {vp_r / (vp_r + fn_r):.3f}   precisao {vp_r / max(1, vp_r + fp_r):.3f}")
print(f"custo medio por cliente: R$ {custos_r.min() / n_te:.2f}"
      f"   (no corte 0,5: R$ {custo_total(0.5, c_fp_r, c_fn_r) / n_te:.2f})")

O corte teórico é $30/(30+900) = 0{,}0323$, e a grade encontra $0{,}0273$ — dois
passos de distância, num trecho em que a curva de custo é plana.

Com ele, **por mil clientes**: 578 investigados, dos quais 92,7 são positivos de
verdade e 486 são falsos positivos. Existiam 98,5 positivos, então o modelo encontra
$94{,}1\%$ deles.

Vale ler as duas taxas juntas. A **revocação é $0{,}941$** — quase todos os
positivos são pescados. A **precisão é $0{,}160$** — de cada seis pessoas
incomodadas, uma tinha o problema. Isso soa péssimo até você fazer a conta que a
motivou: cada positivo encontrado vale R$ 900 de prejuízo evitado, e cada
investigação custa R$ 30. Incomodar cinco pessoas à toa para encontrar uma vale a
pena por uma ordem de grandeza.

O custo médio cai de **R$ 64,14 para R$ 19,82 por cliente**, uma redução de 69%,
sem trocar de modelo e sem coletar um dado a mais — só movendo um número que o
`predict` esconde atrás do $0{,}5$.

E é aqui que a Seção anterior cobra o seu preço: esse corte só é ótimo se a
probabilidade estimada for **calibrada**. Se o modelo diz $0{,}03$ e a frequência
real naquele grupo é $0{,}10$, a conta de custo esperado está errada, e o corte
também. É o assunto da próxima seção.

---
## 6. ROC e AUC: avaliando sem escolher o corte

Se o corte é uma decisão de negócio, faz sentido uma métrica que avalie o modelo
**para todos os cortes ao mesmo tempo**. A curva ROC põe a taxa de verdadeiros
positivos contra a taxa de falsos positivos, varrendo o corte; a AUC é a área
debaixo dela.

A AUC tem uma interpretação exata e pouco divulgada:

$$\mathrm{AUC} = P\big(\widehat p(X^+) > \widehat p(X^-)\big),$$

a probabilidade de o modelo dar nota maior a um positivo sorteado ao acaso do que a
um negativo sorteado ao acaso. Vamos conferir por simulação.

In [ ]:
fpr, tpr, cortes_roc = roc_curve(y_te, p)
auc = roc_auc_score(y_te, p)

rng = np.random.default_rng(0)
pos = p[y_te == 1]
neg = p[y_te == 0]
sorteio = 200_000
venceu = pos[rng.integers(0, len(pos), sorteio)] > neg[rng.integers(0, len(neg), sorteio)]

print(f"AUC calculada pelo sklearn        : {auc:.4f}")
print(f"P(nota do positivo > nota do negativo), por sorteio: {venceu.mean():.4f}")

In [ ]:
fig, ax = subplots(figsize=(5.0, 4.2))
ax.plot(fpr, tpr, lw=1.6, label=f"logistica (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], ls="--", color="gray", label="palpite aleatorio (AUC = 0,5)")
for alvo, marca in [(0.5, "o"), (0.2, "s"), (0.05, "^")]:
    j = int(np.argmin(np.abs(cortes_roc - alvo)))
    ax.plot(fpr[j], tpr[j], marca, color="crimson", ms=6)
    ax.annotate(f"corte {alvo}", xy=(fpr[j], tpr[j]), xytext=(8, -10),
                textcoords="offset points", fontsize=7.5, color="crimson")
ax.set_xlabel("taxa de falsos positivos"); ax.set_ylabel("taxa de verdadeiros positivos")
ax.legend(fontsize=8, loc="lower right")

Repare onde caem os três cortes marcados. O $0{,}5$ fica encolhido no canto
inferior esquerdo — quase nenhum falso positivo, e quase nenhum verdadeiro positivo
também. É a mesma informação da Seção 2, agora geométrica: com a classe rara, o
corte padrão quase não classifica ninguém como positivo.

A AUC não depende do corte **nem da prevalência**, o que é uma virtude e um defeito.
Virtude: você compara modelos sem antes decidir a política. Defeito: ela pode
parecer boa num problema em que qualquer corte útil produz uma precisão sofrível — e
é aí que entra a curva da próxima seção.

---
## 7. Precisão–revocação, a curva certa para classes raras

A ROC usa a taxa de falsos positivos, $FP/(FP+VN)$, cujo denominador é gigantesco
quando a classe negativa domina. Mil falsos positivos entre 36 mil negativos mal
mexem no eixo horizontal — mas arruínam a precisão, que é o que a pessoa que vai
agir sobre a lista realmente sente.

A curva precisão–revocação não tem esse problema, porque nenhum dos dois eixos usa
$VN$.

In [ ]:
pr, rc, _ = precision_recall_curve(y_te, p)
ap = average_precision_score(y_te, p)
prevalencia = y_te.mean()

fig, ax = subplots(figsize=(5.2, 3.4))
ax.plot(rc, pr, lw=1.6, label=f"logistica (AP = {ap:.3f})")
ax.axhline(prevalencia, ls="--", color="gray",
           label=f"palpite aleatorio (= prevalencia = {prevalencia:.3f})")
ax.set_xlabel("revocacao"); ax.set_ylabel("precisao")
ax.legend(fontsize=8)

print(f"AUC (ROC)                     : {auc:.4f}   (linha de base 0,5)")
print(f"precisao media (AP)           : {ap:.4f}   (linha de base {prevalencia:.4f})")
print(f"\nquanto cada uma melhora sobre a linha de base:")
print(f"   ROC: {auc/0.5:.2f}x     PR: {ap/prevalencia:.2f}x")

A linha de base das duas curvas é diferente, e é isso que muda a leitura. Um
palpite aleatório tem AUC $=0{,}5$ **sempre**; já a precisão média de um palpite
aleatório é a **prevalência**. Comparar 0,86 contra 0,5 e comparar 0,3 contra 0,1
são leituras muito diferentes do mesmo modelo.

Regra prática: com classes equilibradas, use a ROC. Com uma classe rara, reporte as
duas — e, se tiver de escolher uma, a precisão–revocação diz mais sobre o que o
usuário do modelo vai viver.

---
## 8. `class_weight`: reponderar é mexer no corte

O `scikit-learn` oferece `class_weight="balanced"`, que dá a cada classe um peso
inversamente proporcional à sua frequência. É frequentemente apresentado como *a*
solução para desbalanceamento. Vale ver o que ele faz de fato.

In [ ]:
balanceado = Pipeline([("sc", StandardScaler()),
                       ("lg", LogisticRegression(max_iter=2000,
                                                 class_weight="balanced"))]).fit(X_tr, y_tr)
p_bal = balanceado.predict_proba(X_te)[:, 1]

linhas = []
for nome, prob, pred_ in [("logistica", p, p >= 0.5),
                          ("logistica balanceada", p_bal, p_bal >= 0.5),
                          ("logistica, corte no F1", p, p >= t_f1)]:
    tp = np.sum(pred_ & (y_te == 1)); fp_ = np.sum(pred_ & (y_te == 0))
    fn_ = np.sum(~pred_ & (y_te == 1))
    linhas.append({"modelo": nome, "AUC": roc_auc_score(y_te, prob),
                   "AP": average_precision_score(y_te, prob),
                   "precisao": tp / (tp + fp_) if tp + fp_ else np.nan,
                   "revocacao": tp / (tp + fn_),
                   "previstos positivos": int(pred_.sum())})
pd.DataFrame(linhas).set_index("modelo").round(4)

A AUC e a AP praticamente não mudam — o `class_weight` **não melhora o
ordenamento**. O que ele faz é deslocar as probabilidades para cima, de modo que o
corte fixo de $0{,}5$ passe a marcar muito mais gente como positiva. Em outras
palavras: é uma maneira indireta de mexer no corte.

Isso não o torna inútil, mas muda o conselho. Se o seu problema é *"o corte padrão
não serve"*, mexer no corte é mais direto, mais transparente e não distorce as
probabilidades estimadas — que é justamente o assunto da próxima seção.

---
## 9. Ordenar bem $\neq$ estimar bem

Chegamos à distinção mais fina da aula. A AUC só olha a **ordem** das
probabilidades: se você elevar todas ao cubo, a AUC não muda. Mas as probabilidades
mudaram, e se alguém for usá-las para calcular um custo esperado, o resultado muda
junto.

Um modelo é **calibrado** quando, entre as observações a que ele deu 30%, cerca de
30% são de fato positivas. Vamos comparar dois modelos numa população em que as
covariáveis são fortemente correlacionadas — o que torna a suposição do Bayes
ingênuo **falsa**.

In [ ]:
def cenario(n, rng, prev=0.08):
    """Classe rara, com covariaveis MUITO correlacionadas entre si."""
    y = (rng.uniform(size=n) < prev).astype(int)
    d = 6
    S = np.full((d, d), 0.85) + np.eye(d) * 0.15
    X = rng.normal(size=(n, d)) @ np.linalg.cholesky(S).T
    X[y == 1] += 1.05
    return X, y


Xc, yc = cenario(4000, np.random.default_rng(43))
Xc_te, yc_te = cenario(20_000, np.random.default_rng(44))

candidatos = [("regressao logistica",
               Pipeline([("sc", StandardScaler()), ("lg", LogisticRegression())])),
              ("Bayes ingenuo", GaussianNB())]

linhas, probs = [], {}
for nome, m in candidatos:
    m.fit(Xc, yc)
    pc = m.predict_proba(Xc_te)[:, 1]
    probs[nome] = pc
    linhas.append({"modelo": nome, "AUC": roc_auc_score(yc_te, pc),
                   "Brier": brier_score_loss(yc_te, pc)})
pd.DataFrame(linhas).set_index("modelo").round(4)

In [ ]:
fig, (ax1, ax2) = subplots(1, 2, figsize=(7.8, 3.2))
for (nome, _), cor in zip(candidatos, ["steelblue", "crimson"]):
    fr, mp = calibration_curve(yc_te, probs[nome], n_bins=12, strategy="quantile")
    ax1.plot(mp, fr, "o-", ms=4, color=cor, label=nome)
    ax2.hist(probs[nome], bins=40, alpha=0.55, color=cor, label=nome)
ax1.plot([0, 1], [0, 1], ls="--", color="gray", label="calibracao perfeita")
ax1.set_xlabel("probabilidade estimada"); ax1.set_ylabel("frequencia observada")
ax1.set_title("curva de calibracao", fontsize=9); ax1.legend(fontsize=7.5)
ax2.set_yscale("log"); ax2.set_xlabel("probabilidade estimada")
ax2.set_title("distribuicao das probabilidades", fontsize=9); ax2.legend(fontsize=7.5)

> **A lição.** As duas AUCs são praticamente iguais: os dois modelos **ordenam
> igualmente bem**. Já o Brier — o erro quadrático médio das probabilidades — do
> Bayes ingênuo é várias vezes maior.
>
> A curva de calibração mostra por quê. O Bayes ingênuo multiplica seis densidades
> como se fossem independentes, quando na verdade elas têm correlação $0{,}85$
> entre si. Multiplicar seis evidências quase redundantes como se fossem seis
> evidências novas produz probabilidades **empurradas para os extremos**: quase tudo
> vira 0 ou 1, como o histograma da direita mostra.
>
> Consequência prática: se você só vai **ranquear** — mandar os mil clientes de
> maior nota para a equipe —, a AUC basta e o Bayes ingênuo serve. Se você vai
> **multiplicar a probabilidade por um valor em reais** para decidir, ela não serve,
> e o corte ótimo da Seção 5 sai errado. Meça o Brier, ou o log-loss, antes de
> confiar num número que vai virar dinheiro.

O `CalibratedClassifierCV` aprende uma transformação **monótona** das
probabilidades num conjunto separado. Monótona é a palavra que importa: ela não
muda a ordem. A pergunta é o que isso faz com cada uma das duas métricas.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

nb_calibrado = CalibratedClassifierCV(GaussianNB(), method="isotonic", cv=5).fit(Xc, yc)
p_cal = nb_calibrado.predict_proba(Xc_te)[:, 1]

comparacao_cal = pd.DataFrame([
    {"modelo": "regressao logistica",
     "AUC": roc_auc_score(yc_te, probs["regressao logistica"]),
     "Brier": brier_score_loss(yc_te, probs["regressao logistica"])},
    {"modelo": "Bayes ingenuo",
     "AUC": roc_auc_score(yc_te, probs["Bayes ingenuo"]),
     "Brier": brier_score_loss(yc_te, probs["Bayes ingenuo"])},
    {"modelo": "Bayes ingenuo calibrado",
     "AUC": roc_auc_score(yc_te, p_cal),
     "Brier": brier_score_loss(yc_te, p_cal)},
]).set_index("modelo")
print(comparacao_cal.round(4).to_string())

fig, ax = subplots(figsize=(5.2, 3.2))
for rotulo, pp, cor in [("Bayes ingenuo", probs["Bayes ingenuo"], "crimson"),
                        ("Bayes ingenuo calibrado", p_cal, "seagreen")]:
    fr, mp = calibration_curve(yc_te, pp, n_bins=12, strategy="quantile")
    ax.plot(mp, fr, "o-", ms=4, color=cor, label=rotulo)
ax.plot([0, 1], [0, 1], ls="--", color="gray", label="calibracao perfeita")
ax.set_xlabel("probabilidade estimada"); ax.set_ylabel("frequencia observada")
ax.legend(fontsize=8)

| modelo | AUC | Brier |
| --- | --- | --- |
| regressão logística | $0{,}7740$ | $0{,}0653$ |
| Bayes ingênuo | $0{,}7762$ | $0{,}1523$ |
| Bayes ingênuo calibrado | $0{,}7758$ | $0{,}0653$ |

**A AUC não se move e o Brier despenca.** De $0{,}7762$ para $0{,}7758$ — quatro
décimos de milésimo, e para *baixo* — contra um Brier que cai de $0{,}1523$ para
$0{,}0653$, menos da metade. E $0{,}0653$ é exatamente o Brier da logística: depois
de calibrado, o Bayes ingênuo estima probabilidades tão boas quanto as dela.

O motivo é a palavra do enunciado. A calibração isotônica é **monótona**: ela pode
mapear $0{,}999 \mapsto 0{,}22$ e $0{,}001 \mapsto 0{,}03$, mas nunca inverte duas
observações. A AUC só olha a ordem, então não tem como mudar — a queda de
$0{,}0004$ vem dos empates que a isotônica cria ao achatar faixas inteiras no mesmo
valor. O Brier olha a distância até $0$ ou $1$, e é tudo o que a transformação
conserta.

**A separação entre as duas métricas fica exposta.** As três linhas têm
praticamente a mesma AUC e dois valores de Brier muito diferentes. Ordenar bem e
estimar bem são propriedades distintas, e a segunda é recuperável a partir da
primeira: dado um modelo que ordena bem, uma transformação monótona ajustada num
conjunto separado entrega as probabilidades.

Na prática: se o Bayes ingênuo é o que você tem e você precisa de probabilidades —
para a conta de custo da Seção 5, por exemplo —, não troque de modelo. Calibre.

---
## 10. O `scoring` da validação cruzada

Tudo o que vimos desaba se a busca de hiperparâmetros otimizar a métrica errada — e
o padrão do `GridSearchCV` em classificação é a **acurácia**, exatamente a métrica
que esta aula inteira desaconselha.

In [ ]:
sub = np.random.default_rng(0).choice(len(y_tr), 12_000, replace=False)
X_s, y_s = X_tr[sub], y_tr[sub]

base = Pipeline([("sc", StandardScaler()), ("lg", LogisticRegression(max_iter=2000))])
grade = {"lg__C": np.logspace(-4, 2, 7)}

resultados = {}
for criterio in ["accuracy", "roc_auc", "average_precision"]:
    b = skm.GridSearchCV(base, grade, cv=4, scoring=criterio, n_jobs=-1).fit(X_s, y_s)
    pv = b.predict_proba(X_te)[:, 1]
    resultados[criterio] = {"C escolhido": b.best_params_["lg__C"],
                            "acuracia": (pv >= 0.5).astype(int).__eq__(y_te).mean(),
                            "AUC": roc_auc_score(y_te, pv),
                            "AP": average_precision_score(y_te, pv)}
pd.DataFrame(resultados).T.round(4)

Os três critérios não concordam. Otimizar acurácia escolhe um $C$ mil vezes maior
e entrega um modelo que **ordena pior** — AUC e AP mais baixas. E repare no outro
lado: os dois critérios que olham ordenamento escolhem uma regularização
fortíssima, cuja acurácia é exatamente a do classificador trivial da Seção 2 — ele
não chama ninguém de positivo no corte $0{,}5$, e nem por isso é um modelo ruim,
porque a nota que ele atribui ordena melhor.

É a lição da aula inteira em uma tabela: **acurácia e ordenamento são objetivos
diferentes, e otimizar um pode custar o outro.** A regra é direta: **passe para o
`scoring` a métrica pela qual você vai ser julgado.** `"roc_auc"` para ordenamento, `"average_precision"` para
classes raras, `"f1"` quando precisão e revocação importam igualmente,
`"neg_brier_score"` ou `"neg_log_loss"` quando as probabilidades vão virar decisão.

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| acurácia | §2 | com prevalência de 10%, o modelo empata com "ninguém é positivo" |
| precisão × revocação | §3 | perguntas diferentes; quem age quer precisão, quem teme perder quer revocação |
| o corte | §4 | $0{,}5$ não maximiza o $F_1$ — nem tinha por que |
| custo assimétrico | §5 | o corte ótimo é $c_{FP}/(c_{FP}+c_{FN})$, e o medido bate com o teórico |
| AUC | §6 | é $P(\widehat p(X^+) > \widehat p(X^-))$ — confirmado por sorteio |
| ROC × PR | §7 | a linha de base da ROC é sempre $0{,}5$; a da PR é a prevalência |
| `class_weight` | §8 | não melhora o ordenamento; desloca o corte por outro caminho |
| calibração | §9 | mesma AUC, Brier muito diferente — ordenar bem não é estimar bem |
| `scoring` | §10 | o padrão é acurácia, justamente a métrica que esta aula desaconselha |

**Leitura recomendada.** [AME] §7.4 (métricas para classificação) e §9.1–9.2 (o
risco 0–1 e as perdas assimétricas, de onde sai a fórmula da Seção 5). [ISLP] §4.4.2
(a matriz de confusão no exemplo do `Default`, com a mesma discussão de sensibilidade
e especificidade) e §4.4.3 (a curva ROC). O material `MatConf.pdf`, na pasta desta
aula, é um resumo de bolso das definições.

**Para praticar.** `Lista teorica 08.pdf` (teórica, com gabarito) e
`Lista prática 08.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** A Aula 09 traz um classificador construído sobre uma ideia geométrica
diferente — a margem — e que, por não estimar probabilidades, ilustra bem a
distinção da Seção 9.